# DATA304 · ColBERT ANN Top-900을 재정렬하는 Cross-Attention Scorer

Epoch 5 ColBERT의 리뷰·상품 토큰과 기존 FAISS IVF를 고정한다. Gold TRAIN에서 단방향 cross-attention과 두 MLP를 하나의 listwise ranking loss로 학습한다. 각 epoch의 선택 기준은 같은 VAL 350개에서 측정한 **Recall@100**이다.

SageMaker 프로젝트 폴더의 기존 PyTorch 커널에서 위에서 아래로 실행한다. 마지막 셀이 학습을 시작한다. 검증 결과의 `attention_top100_item_ids`는 기존 Cross-Encoder에 전달할 수 있는 순서 있는 상품 ID 목록이다.

## 1. 경로와 실험 설정

기존 `data304_outputs/additional_training/epoch_05/checkpoints/colbert_gold.pt`와 해당 epoch의 인덱스를 사용한다. 기본 설정은 ANN의 토큰당 이웃 1,024개, IVF 탐색 셀 32개, 후보 900개다. 후보 micro-batch는 한 쿼리에서 동시에 채점할 상품 수다.

In [ ]:
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import fcntl
from collections import Counter, defaultdict
from contextlib import contextmanager, nullcontext
from copy import deepcopy
from datetime import datetime
from functools import lru_cache
from itertools import chain
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

# 기존 PyTorch 커널에서 없는 검색 라이브러리만 설치한다. 리뷰는 이 컴퓨터에서만 처리한다.
if importlib.util.find_spec('faiss') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'numpy=={np.__version__}', 'faiss-cpu'])
if importlib.util.find_spec('transformers') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers>=4.48,<5'])
import faiss
from transformers import AutoConfig, AutoModel, AutoTokenizer

# SageMaker에서는 노트북을 프로젝트 폴더에 둔다. 다른 위치에서 열면 PROJECT_DIR만 지정한다.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'catalog.json').is_file():
    PROJECT_DIR = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
COLBERT_EPOCH = 5
ANN_NPROBE, ANN_PER_TOKEN, ANN_POOL = 32, 1024, 900
RERANK_TOPK, VAL_LIMIT = 100, 350
EMBED_DIM, NUM_ATTENTION_HEADS, ATTENTION_DROPOUT = 64, 4, 0.1
EPOCHS, LEARNING_RATE, WEIGHT_DECAY = 3, 3e-4, 0.01
MAX_GRAD_NORM, RANK_TEMPERATURE, SEED = 1.0, 1.0, 42
NEGATIVE_COUNT = 8
TRAIN_QUERY_BATCH, SCORER_CANDIDATE_BATCH, GRAD_ACCUM_STEPS = 1, 16, 4
CACHE_QUERIES = 50  # 완료한 쿼리 묶음을 재시작 시 바로 읽는다.
WEIGHT_SAMPLE_QUERIES = 5  # 고정 VAL 순서의 앞 5개 쿼리에서 후보 900개 모두의 리뷰별 가중치를 저장한다.
INDEX_OVERRIDE = None  # 기존 epoch 5 인덱스를 옮긴 경우 해당 폴더의 Path를 지정한다.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
faiss.omp_set_num_threads(min(8, os.cpu_count() or 4))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
if (
    min(
        EPOCHS,
        LEARNING_RATE,
        RANK_TEMPERATURE,
        TRAIN_QUERY_BATCH,
        SCORER_CANDIDATE_BATCH,
        GRAD_ACCUM_STEPS,
        CACHE_QUERIES,
        MAX_GRAD_NORM,
    )
    <= 0
    or EMBED_DIM % NUM_ATTENTION_HEADS
    or NEGATIVE_COUNT != 8
    or not RERANK_TOPK <= 200 <= ANN_POOL
):
    raise ValueError('학습 설정, 후보 수 및 attention head 차원을 확인하세요.')
print('프로젝트', PROJECT_DIR, '| 장치', DEVICE, '| BF16', USE_BF16, flush=True)

## 2. Gold 분할과 카탈로그

저장된 `split.json`의 TRAIN 전체와 앞쪽 VAL 350개를 원래 순서로 복원한다. 상품 인덱스의 행 번호는 `catalog.json` 순서다. 카테고리 오답을 고를 때 쓰는 상품–카테고리 연결은 TRAIN에서 가져온다.

In [ ]:
# 아래 기본 읽기·정규화 함수는 기존 DATA304 노트북과 같다.
def read_json(path):
    # 기존 JSON 파일의 UTF-8 BOM 유무를 모두 처리한다.
    return json.loads(path.read_text(encoding='utf-8-sig'))


def write_json(path, value):
    # 완성된 설정을 나중에 점수표와 대조할 수 있도록 사람이 읽을 수 있는 형식으로 쓴다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


@lru_cache(maxsize=32)
def file_hash(path, size, modified_ns):
    # 모델과 입력이 같을 때만 캐시를 공유한다. 같은 실행에서 큰 파일을 매번 다시 읽지 않는다.
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def sha256(path):
    # 파일을 교체했으면 크기 또는 수정 시각이 바뀌므로 새 해시를 얻는다.
    stat = path.stat()
    return file_hash(path, stat.st_size, stat.st_mtime_ns)


def signature(value):
    # 파일 내용과 설정이 일치하는 후보 캐시·실험을 찾기 위한 해시를 만든다.
    data = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(data).hexdigest()


def load_pt(path):
    # CPU 메모리 매핑으로 저장한 모델과 AdamW에서 필요한 텐서부터 읽는다.
    return torch.load(path, map_location='cpu', mmap=True, weights_only=True)


def amp_context():
    # 원래 실행처럼 지원되는 CUDA에서는 bfloat16으로 순전파한다.
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def metadata_text(item):
    # 학습 때 사용한 네 필드의 이름, 순서, [SEP]를 그대로 지켜 임베딩 입력을 맞춘다.
    fields = [
        ('title', 'Product'),
        ('store', 'Store'),
        ('features', 'Features'),
        ('description', 'Description'),
    ]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


def gold_query(row):
    # 리뷰 다섯 개는 원래 순서를 지킨다. Gold 카테고리 경로는 기존 방식의 오답을 뽑을 때 사용한다.
    if len(row['reviews']) != 5:
        raise ValueError(f'리뷰가 5개가 아닌 쿼리 {row["query_id"]}')
    return {
        'category_path': tuple(
            level.strip() for level in (row.get('category_path') or '').split('>') if level.strip()
        ),
        'query_id': row['query_id'],
        'item_id': row['item_id'],
        'reviews': [review['text'].strip() for review in row['reviews']],
    }


def atomic_file(path, writer):
    """중간에 중단된 파일이 완성된 결과로 보이지 않게 같은 파일시스템에서 교체한다."""
    with TemporaryDirectory(dir=path.parent) as tmp:
        stage = Path(tmp) / path.name
        writer(stage)
        stage.replace(path)


def log_line(log, text, echo=False):
    """긴 후보 준비와 학습을 tail로 확인할 수 있게 매 줄을 즉시 기록한다."""
    line = f'{datetime.now().astimezone().isoformat(timespec="seconds")} | {text}'
    print(line, file=log, flush=True)
    if echo:
        print(line, flush=True)


@contextmanager
def file_lock(path, blocking=True):
    """동일한 실험의 중복 실행과 캐시를 동시에 만드는 두 커널의 충돌을 막는다."""
    with path.open('a+') as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | (0 if blocking else fcntl.LOCK_NB))
        except BlockingIOError as exc:
            raise RuntimeError(f'같은 실험이 실행 중입니다. {path}') from exc
        yield


RUN_DIR = PROJECT_DIR / 'data304_outputs'
ROOT = RUN_DIR / 'colbert_attention_reranker'
CKPT_SOURCE_DIR = RUN_DIR / 'additional_training' / f'epoch_{COLBERT_EPOCH:02d}'
SOURCE = CKPT_SOURCE_DIR / 'checkpoints' / 'colbert_gold.pt'
CATALOG_PATH, GOLD_PATH = PROJECT_DIR / 'catalog.json', PROJECT_DIR / 'train_gold.json'
SPLIT_PATH = RUN_DIR / 'split.json'
for path in (CATALOG_PATH, GOLD_PATH, SPLIT_PATH, SOURCE):
    if not path.is_file():
        raise FileNotFoundError(f'기존 프로젝트 입력을 찾을 수 없습니다. {path}')
CFG = {}
for path in (RUN_DIR / 'aspect_vocabulary_and_config.json', RUN_DIR / 'run_manifest.json'):
    if path.is_file():
        CFG.update(read_json(path)['config'])
if not CFG or (CFG['query_tokens'], CFG['metadata_tokens'], CFG['dimension']) != (72, 112, EMBED_DIM):
    raise ValueError('epoch 5의 config에서 query_tokens=72, metadata_tokens=112, dimension=64를 확인하세요.')
SOURCE_SHA = sha256(SOURCE)
catalog = read_json(CATALOG_PATH)
ITEM_IDS = [d['item_id'] for d in catalog]
ITEM_TO_IDX = {d: i for i, d in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(d) for d in catalog]
del catalog
gold = read_json(GOLD_PATH)
by_id = {r['query_id']: gold_query(r) for r in gold}
if len(by_id) != len(gold) or len(ITEM_TO_IDX) != len(ITEM_IDS) or len(ITEM_IDS) < ANN_POOL:
    raise ValueError('Gold 쿼리 ID와 카탈로그 상품 ID의 유일성 및 카탈로그 크기를 확인하세요.')
del gold
split = read_json(SPLIT_PATH)
train_ids, val_ids = split['train_query_ids'], split['val_query_ids']
if (
    not train_ids
    or not 0 < VAL_LIMIT <= len(val_ids)
    or len(set(train_ids + val_ids)) != len(train_ids + val_ids)
    or not set(train_ids + val_ids) <= by_id.keys()
):
    raise ValueError('기존 split.json의 쿼리 및 순서를 확인하세요.')
TRAIN = [by_id[q] for q in train_ids]
full_val = [by_id[q] for q in val_ids]
if any(r['item_id'] not in ITEM_TO_IDX for r in chain(TRAIN, full_val)) or {r['item_id'] for r in TRAIN} & {
    r['item_id'] for r in full_val
}:
    raise ValueError('Gold 상품이 카탈로그에 없거나 TRAIN·VAL에 걸쳐 있습니다.')
VAL = full_val[:VAL_LIMIT]
del full_val, by_id
# 카테고리 오답의 사전은 TRAIN 연결만으로 만든다.
TRAIN_CATEGORY = {ITEM_TO_IDX[r['item_id']]: r['category_path'] for r in TRAIN if r['category_path']}
CAT_TO_DOCS, PARENT_TO_DOCS = defaultdict(list), defaultdict(list)
for doc, path in TRAIN_CATEGORY.items():
    CAT_TO_DOCS[path].append(doc)
    PARENT_TO_DOCS[path[:-1]].append(doc)
CACHE_DIR = ROOT / 'caches'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'TRAIN {len(TRAIN):,} · VAL {len(VAL):,} · 상품 {len(ITEM_IDS):,}\n고정 가중치 {SOURCE}')

## 3. 고정 인코더와 상품 토큰

체크포인트 SHA-256, 카탈로그, 인코딩 설정이 일치하는 기존 epoch 5 인덱스를 찾는다. 상품 벡터나 마스크가 없으면 같은 체크포인트로 새 `caches` 아래에 복구한다. IVF 자체가 유실된 경우에는 기존 `token_ivf.index`를 복원해야 같은 ANN을 쓸 수 있다. `INDEX_OVERRIDE`는 확인된 기존 인덱스를 다른 경로에 옮겼을 때 사용한다.

ANN은 각 리뷰 토큰이 찾은 상품별 최대 유사도의 양수 부분을 누적한다. 점수가 같으면 카탈로그 행 번호를 사용한다. 기준 모델인 Exact MaxSim은 각 리뷰 토큰이 얻은 최대 내적을 합산하고 5개 리뷰에 대해 평균한다.

In [ ]:
class ColBERTRetriever(nn.Module):
    def __init__(self, config, dimension):
        super().__init__()
        self.query_encoder = AutoModel.from_config(config)
        self.doc_encoder = AutoModel.from_config(config)
        self.query_proj = nn.Linear(config.hidden_size, dimension, bias=False)
        self.doc_proj = nn.Linear(config.hidden_size, dimension, bias=False)

    # 학습 노트북과 같은 정규화와 유효 토큰 마스크를 사용해 MaxSim용 벡터를 반환한다.
    def encode(self, tokens, side):
        backbone = self.query_encoder if side == 'query' else self.doc_encoder
        projection = self.query_proj if side == 'query' else self.doc_proj
        hidden = backbone(**tokens).last_hidden_state
        mask = tokens['attention_mask'].bool()
        special = torch.zeros_like(mask)
        for token_id in SPECIAL_IDS:
            special |= tokens['input_ids'].eq(token_id)
        valid = mask & ~special
        empty = ~valid.any(dim=-1)
        if empty.any():
            valid[empty] = mask[empty]
        return F.normalize(projection(hidden), p=2, dim=-1), valid


def tokenize(texts, max_length):
    out = COL_TOKENIZER(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    return {k: v.to(DEVICE) for k, v in out.items()}


def sync_device():
    """CUDA의 비동기 연산이 끝난 시점에 후보 채점 시간을 잰다."""
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)


def resolve_index():
    """기존 학습·하드오답 노트북의 epoch 5 인덱스 서명과 맞는 훈련된 IVF만 선택한다."""
    params = {
        k: CFG[k]
        for k in (
            'backbone',
            'dimension',
            'metadata_tokens',
            'ann_nlist',
            'index_doc_batch',
            'index_add_docs',
        )
    }
    current = 'bfloat16' if USE_BF16 else 'float32'
    specs = [
        dict(
            weights=SOURCE_SHA,
            catalog=sha256(CATALOG_PATH),
            seed=int(split['seed']),
            precision=p,
            config=params,
        )
        for p in dict.fromkeys((current, 'bfloat16', 'float32'))
    ]
    roots = [CKPT_SOURCE_DIR / 'indexes', RUN_DIR / 'cross_hard4_mixed4_lr5e-6' / 'indexes']
    tried = []
    for spec in specs:
        key = signature(spec)[:20]
        choices = [Path(INDEX_OVERRIDE)] if INDEX_OVERRIDE is not None else [root / key for root in roots]
        matches = []
        for folder in choices:
            tried.append(str(folder))
            info = folder / 'index_info.json'
            # 일부 기존 노트북은 manifest 없이 설정 해시를 디렉터리 이름에만 기록했다.
            match = read_json(info) == spec if info.is_file() else folder.name == key
            if match and folder.is_dir():
                matches.append(folder)
        ann = next((d for d in matches if (d / 'token_ivf.index').is_file()), None)
        if ann is not None:
            return spec, ann, matches
    raise FileNotFoundError(
        '같은 epoch 5의 학습된 token_ivf.index가 필요합니다. 기존 인덱스를 복원하거나 '
        'INDEX_OVERRIDE에 그 위치를 지정하세요.\n' + '\n'.join(dict.fromkeys(tried))
    )


def load_encoder():
    """기존 2개 인코더·투영층만 엄격히 복원하고 모든 파라미터를 고정한다."""
    model = ColBERTRetriever(AutoConfig.from_pretrained(CFG['backbone']), EMBED_DIM)
    state = load_pt(SOURCE)
    model.load_state_dict({k: state[k] for k in model.state_dict()}, strict=True)
    return model.to(DEVICE).eval().requires_grad_(False)


@torch.inference_mode()
def regenerate_documents(encoder, folder):
    """기존 인덱스 생성과 같은 순서·정밀도로 상품 벡터와 마스크 두 파일만 새 캐시에 기록한다."""
    shape = (len(ITEM_IDS), CFG['metadata_tokens'], EMBED_DIM)
    em = np.lib.format.open_memmap(folder / 'doc_embeddings.npy', mode='w+', dtype=np.float16, shape=shape)
    ma = np.lib.format.open_memmap(folder / 'doc_masks.npy', mode='w+', dtype=np.bool_, shape=shape[:2])
    batch = CFG['index_doc_batch']
    for start in range(0, len(ITEM_IDS), batch):
        stop = min(start + batch, len(ITEM_IDS))
        tokens = tokenize(CAT_TEXTS[start:stop], shape[1])
        with (
            torch.autocast('cuda', dtype=torch.bfloat16)
            if INDEX_SPEC['precision'] == 'bfloat16'
            else nullcontext()
        ):
            vector, mask = encoder.encode(tokens, 'doc')
        width = vector.shape[1]
        em[start:stop] = 0
        ma[start:stop] = False
        em[start:stop, :width] = vector.float().cpu().numpy().astype(np.float16)
        ma[start:stop, :width] = mask.cpu().numpy()
    em.flush()
    ma.flush()


def open_frozen_features(encoder, log):
    """검증된 기존 상품 파일을 재사용한다. 빠진 벡터와 마스크는 새 폴더에만 복구한다."""
    recovered = CACHE_DIR / ('documents_' + signature([INDEX_SPEC, TOKEN_SPEC])[:20])
    paths = [*MATCHED_DIRS, recovered]

    def has_docs(folder):
        return all((folder / n).is_file() for n in ('doc_embeddings.npy', 'doc_masks.npy'))

    with file_lock(CACHE_DIR / '.documents.lock'):
        docs = next((p for p in paths if has_docs(p)), None)
        if docs is None:
            if INDEX_SPEC['precision'] == 'bfloat16' and not USE_BF16:
                raise RuntimeError('BF16으로 만든 인덱스의 상품 벡터 복구에는 BF16 CUDA 장치가 필요합니다.')
            log_line(log, f'상품 토큰·마스크를 epoch 5에서 복구 | {recovered}', True)
            with TemporaryDirectory(dir=CACHE_DIR) as tmp:
                staged = Path(tmp) / 'documents'
                staged.mkdir()
                regenerate_documents(encoder, staged)
                write_json(staged / 'source.json', {'index': INDEX_SPEC, 'tokenizer': TOKEN_SPEC})
                staged.rename(recovered)
            docs = recovered
    if docs == recovered and read_json(docs / 'source.json') != {
        'index': INDEX_SPEC,
        'tokenizer': TOKEN_SPEC,
    }:
        raise ValueError(f'복구한 상품 캐시의 출처가 다릅니다. {docs}')
    em = np.load(docs / 'doc_embeddings.npy', mmap_mode='r', allow_pickle=False)
    ma = np.load(docs / 'doc_masks.npy', mmap_mode='r', allow_pickle=False)
    if (
        em.shape != (len(ITEM_IDS), CFG['metadata_tokens'], EMBED_DIM)
        or ma.shape != em.shape[:2]
        or em.dtype != np.float16
        or ma.dtype != np.bool_
        or not ma.any(1).all()
    ):
        raise ValueError(f'상품 임베딩과 마스크의 크기·자료형·유효 길이를 확인하세요. {docs}')
    index = faiss.read_index(str(IVF_DIR / 'token_ivf.index'))
    if (
        not index.is_trained
        or index.d != EMBED_DIM
        or getattr(index, 'nlist', 0) < ANN_NPROBE
        or index.metric_type != faiss.METRIC_INNER_PRODUCT
        or index.ntotal != int(ma.sum())
    ):
        raise ValueError('기존 IVF의 차원·내적·훈련 상태·토큰 수·nprobe를 확인하세요.')
    index.nprobe = ANN_NPROBE
    # 원본은 카탈로그 행 순서대로 유효 토큰을 add했다. 대응표를 잃어도 같은 순서를 복원할 수 있다.
    expected = np.repeat(np.arange(len(ITEM_IDS), dtype=np.int32), ma.sum(1))
    mapping = next(
        (p / 'token_to_item.npy' for p in MATCHED_DIRS if (p / 'token_to_item.npy').is_file()), None
    )
    token2doc = np.load(mapping, mmap_mode='r', allow_pickle=False) if mapping else expected
    if not np.array_equal(token2doc, expected):
        raise ValueError('저장한 토큰-상품 순서가 현재 상품 마스크와 다릅니다.')
    log_line(log, f'고정 IVF {IVF_DIR} | 상품 특징 {docs} | nprobe {index.nprobe}', True)
    return dict(index=index, token2doc=token2doc, emb=em, mask=ma), docs


@torch.inference_mode()
def encode_five_reviews(model, reviews):
    tokens = tokenize(reviews, CFG['query_tokens'])
    with amp_context():
        embeddings, valid = model.encode(tokens, 'query')
    return embeddings.float().cpu().numpy(), valid.cpu().numpy()


def token_hit_order(query, mask, bundle, per_token):
    # 같은 (리뷰 토큰, 상품)에서 최대 유사도 하나를 고른 뒤 양수 부분을 상품별로 합산한다.
    # 정렬·집계를 NumPy로 처리해 TRAIN 전체에서 토큰 이웃을 파이썬 반복문으로 순회하는 비용을 줄인다.
    tokens = np.ascontiguousarray(query[mask], dtype=np.float32)
    distances, hits = bundle['index'].search(tokens, per_token)
    rows, columns = np.nonzero(hits >= 0)
    if rows.size == 0:
        return []
    docs = bundle['token2doc'][hits[rows, columns]]
    keys = rows.astype(np.int64) * len(ITEM_IDS) + docs
    order = np.argsort(keys, kind='stable')
    keys = keys[order]
    starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1]
    maxima = np.maximum.reduceat(distances[rows, columns][order], starts)
    docs = keys[starts] % len(ITEM_IDS)
    rough = np.bincount(docs, weights=np.maximum(maxima, 0.0), minlength=len(ITEM_IDS))
    touched = np.unique(docs)
    return touched[np.argsort(-rough[touched], kind='stable')[:ANN_POOL]].tolist()


@torch.inference_mode()
def maxsim_scores(query, query_mask, bundle, docs):
    """기존 Exact MaxSim을 FP32/BF16 원래 경로로 채점하고 다섯 리뷰의 점수를 평균한다."""
    q = torch.as_tensor(query, device=DEVICE)
    mask = torch.as_tensor(query_mask, device=DEVICE)

    def score_block(sub):
        # OOM 시 이 호출의 임시 텐서가 해제된 뒤 작은 상품 묶음으로 재시도한다.
        d = torch.from_numpy(bundle['emb'][sub].astype(np.float32)).to(DEVICE)
        dm = torch.from_numpy(bundle['mask'][sub]).to(DEVICE)
        with amp_context():
            dots = torch.einsum('rqd,cmd->rcqm', q, d)
            vals = dots.masked_fill(~dm[None, :, None, :], -1e4).amax(-1)
            out = (vals * mask[:, None, :]).sum(-1).mean(0)
        return out.float().cpu().numpy()

    batch, scores, start = 96, [], 0
    while start < len(docs):
        sub = docs[start : start + batch]
        try:
            scores.append(score_block(sub))
            start += len(sub)
            continue
        except torch.cuda.OutOfMemoryError:
            if DEVICE.type != 'cuda' or batch == 1:
                raise
        batch = max(1, batch // 2)
        torch.cuda.empty_cache()
    return np.concatenate(scores)


INDEX_SPEC, IVF_DIR, MATCHED_DIRS = resolve_index()
COL_TOKENIZER = AutoTokenizer.from_pretrained(CFG['backbone'], use_fast=True)
SPECIAL_IDS = set(COL_TOKENIZER.all_special_ids)
tokenizer_json = json.loads(COL_TOKENIZER.backend_tokenizer.to_str())
tokenizer_json.pop('padding', None)
tokenizer_json.pop('truncation', None)
TOKEN_SPEC = dict(
    backbone=CFG['backbone'],
    tokenizer_sha256=signature(tokenizer_json),
    special_ids=sorted(SPECIAL_IDS),
    padding=COL_TOKENIZER.padding_side,
    truncation=COL_TOKENIZER.truncation_side,
    query_tokens=CFG['query_tokens'],
    metadata_tokens=CFG['metadata_tokens'],
    dimension=EMBED_DIM,
)
print('검증된 epoch 5 인덱스', IVF_DIR, flush=True)

## 4. 쿼리 캐시와 오답 8개

각 행에 원래 ANN 후보 900개와 그 순서에 대응하는 Exact MaxSim 점수를 저장한다. 이전 Cross-Encoder의 Top-100 캐시는 나머지 800개를 복원할 수 없어 이 단계에 사용할 수 없다. 쿼리 토큰은 FP32로 보관해 원래 MaxSim 수치를 유지한다. 저장을 마친 묶음은 재시작·다른 Scorer 실험에서 재사용한다.

| TRAIN 슬롯 | 뽑는 방법 |
|---|---|
| Hard 4 | 실제 ANN 900개에서 MaxSim 점수가 가장 높은 오답 4개 |
| ANN 2 | 원래 ANN 순위의 상·하위 절반에서 1개씩 |
| Random 1 | 카탈로그 전체에서 균등 추출 |
| Category 1 | TRAIN의 같은 leaf 또는 같은 부모의 다른 leaf에서 추출. 둘 다 비면 무작위 보충 |

Gold는 TRAIN의 첫 열에 항상 배치한다. VAL의 후보는 실제 ANN이 반환한 900개로 고정한다.

In [ ]:
# 각 행은 쿼리 특징, 실제 ANN 순서, 그 순서에 대응하는 MaxSim 점수를 함께 보관한다.
ROW_DTYPE = np.dtype(
    [
        ('query', '<f4', (5, CFG['query_tokens'], EMBED_DIM)),
        ('mask', '?', (5, CFG['query_tokens'])),
        ('ann', '<i4', (ANN_POOL,)),
        ('maxsim', '<f4', (ANN_POOL,)),
        ('maxsim_seconds', '<f8'),
    ]
)


@lru_cache(maxsize=16)
def mapped_rows(path):
    """무작위 TRAIN 순서에서도 최근 묶음만 mmap으로 열어 파일 핸들 수를 제한한다."""
    return np.load(path, mmap_mode='r', allow_pickle=False)


class QueryCache:
    """split.json의 순서를 유지하는 읽기 전용 특징 테이블이다."""

    def __init__(self, shards):
        self.shards = shards

    def __getitem__(self, i):
        block, offset = divmod(int(i), CACHE_QUERIES)
        return mapped_rows(self.shards[block])[offset]


def prepare_queries(name, rows, encoder, bundle, feature_spec, log):
    """정답 ID에 접근하지 않고 리뷰에서 실제 ANN 900개를 검색한다. 같은 묶음은 재인코딩하지 않는다."""
    query_ids = [r['query_id'] for r in rows]
    identity = signature(dict(features=feature_spec, split=name, ordered_ids=query_ids, chunk=CACHE_QUERIES))
    folder = CACHE_DIR / f'{name}_{identity[:24]}'
    folder.mkdir(parents=True, exist_ok=True)
    shards, reused = [], 0
    with file_lock(folder / '.prepare.lock'):
        for start in range(0, len(rows), CACHE_QUERIES):
            end = min(start + CACHE_QUERIES, len(rows))
            dst = folder / f'{start:06d}_{end:06d}'
            expected = dict(cache_id=identity, query_ids=query_ids[start:end])
            if not dst.exists():
                with TemporaryDirectory(dir=folder) as tmp:
                    staged = Path(tmp) / 'block'
                    staged.mkdir()
                    arr = np.lib.format.open_memmap(
                        staged / 'rows.npy', mode='w+', dtype=ROW_DTYPE, shape=(end - start,)
                    )
                    for offset, row in enumerate(rows[start:end]):
                        q, mask = encode_five_reviews(encoder, row['reviews'])
                        if (
                            q.shape != (*mask.shape, EMBED_DIM)
                            or mask.shape[0] != 5
                            or not 0 < mask.shape[1] <= CFG['query_tokens']
                            or not mask.any(1).all()
                        ):
                            raise ValueError(f'리뷰 토큰 모양/마스크 오류: {row["query_id"]}')
                        ann = np.asarray(token_hit_order(q, mask, bundle, ANN_PER_TOKEN), dtype=np.int32)
                        if len(ann) != ANN_POOL or len(np.unique(ann)) != ANN_POOL:
                            raise ValueError(
                                f'{row["query_id"]}: 실제 ANN 고유 후보 {len(ann)}개. '
                                f'{ANN_POOL}개를 평가하려면 기존 IVF와 검색 설정을 확인하세요.'
                            )
                        sync_device()
                        t0 = time.perf_counter()
                        baseline = maxsim_scores(q, mask, bundle, ann)
                        sync_device()
                        secs = time.perf_counter() - t0
                        if not np.isfinite(baseline).all() or not np.isfinite(q).all():
                            raise FloatingPointError(f'비유한 고정 특징: {row["query_id"]}')
                        record = arr[offset]
                        record['query'] = 0
                        record['mask'] = False
                        record['query'][:, : q.shape[1]] = q
                        record['mask'][:, : q.shape[1]] = mask
                        record['ann'], record['maxsim'], record['maxsim_seconds'] = ann, baseline, secs
                    arr.flush()
                    del arr, record
                    write_json(staged / 'manifest.json', dict(**expected, sha256=sha256(staged / 'rows.npy')))
                    staged.rename(dst)
            else:
                reused += end - start
            path = dst / 'rows.npy'
            meta = read_json(dst / 'manifest.json')
            data = mapped_rows(path)
            if (
                any(meta.get(k) != v for k, v in expected.items())
                or meta.get('sha256') != sha256(path)
                or data.dtype != ROW_DTYPE
                or data.shape != (end - start,)
            ):
                raise ValueError(f'깨졌거나 다른 설정의 쿼리 캐시입니다. {dst}')
            shards.append(path)
            if end == len(rows) or end % (CACHE_QUERIES * 10) == 0 or start == 0:
                log_line(log, f'{name} cache | {end}/{len(rows)} | reused {reused}', True)
    return QueryCache(shards)


def sample_group(record, gold, rng, *, training):
    """정답 열 0, MaxSim 오답 4개, ANN 상·하위 구간 각 1개, 나머지 2개를 순서대로 뽑는다."""
    pool = record['ann']
    order = np.argsort(-record['maxsim'], kind='stable')
    ranked = pool[order]
    hard = ranked[ranked != gold][:4].tolist()
    chosen, used = [gold, *hard], {gold, *hard}
    counts = Counter(maxsim_hard=len(hard))

    def add(doc, label):
        chosen.append(int(doc))
        used.add(int(doc))
        counts[label] += 1

    def draw(items):
        # 구간에 수백 개가 남고 제외 목록은 최대 8개라 전체를 매 쿼리마다 복사하지 않아도 된다.
        doc = int(rng.choice(items))
        while doc in used:
            doc = int(rng.choice(items))
        return doc

    for segment, label in zip(np.array_split(pool, 2), ('ann_upper_half', 'ann_lower_half')):
        add(draw(segment), label)
    if training:
        catalog_indices = range(len(ITEM_IDS))
        add(draw(catalog_indices), 'catalog_random')
        cat = TRAIN_CATEGORY.get(gold)
        options = []
        if cat:
            same = [d for d in CAT_TO_DOCS[cat] if d not in used]
            sibling = [d for d in PARENT_TO_DOCS[cat[:-1]] if d not in used and TRAIN_CATEGORY[d] != cat]
            options = [
                (items, label)
                for items, label in ((same, 'category_same'), (sibling, 'category_sibling'))
                if items
            ]
        if options:
            items, label = rng.choice(options)
            add(rng.choice(items), label)
        else:
            add(draw(catalog_indices), 'category_fallback_random')
    else:
        # VAL loss는 Gold가 실제 ANN 안에 든 쿼리만 사용한다. 나머지 2개도 같은 ANN 내부에서 뽑는다.
        if gold not in pool:
            raise ValueError('VAL loss의 Gold가 실제 ANN에 없습니다.')
        extra = rng.sample([int(d) for d in pool if d not in used], 2)
        for d in extra:
            add(d, 'val_pool_extra')
    if len(used) != 1 + NEGATIVE_COUNT or len(chosen) != len(used):
        raise ValueError('정답 또는 오답이 중복됐습니다.')
    return np.asarray(chosen, dtype=np.int32), counts


def train_group(index, epoch, train_cache):
    """쿼리별 난수를 고정해 셔플·OOM 재시도와 관계없이 해당 epoch의 오답을 재현한다."""
    row = TRAIN[index]
    rng = random.Random(f'{SEED}|train|{epoch}|{row["query_id"]}')
    return sample_group(train_cache[index], ITEM_TO_IDX[row['item_id']], rng, training=True)

## 5. 단방향 Cross-Attention과 두 MLP

리뷰 토큰을 Q, 후보 상품 토큰을 K/V로 사용한다. 각 리뷰의 Q는 해당 상품 토큰에만 attention한다. 다섯 리뷰를 한 연산에 묶고 결과를 다시 리뷰별로 나눈다. 두 MLP는 **같은** `[R, H, |R−H|, R⊙H]`를 받는다. Matching MLP의 유효 토큰 점수 합이 리뷰 점수다. Weight MLP는 유효 토큰의 평균으로 가중치 logit을 만든다. 상품 점수는 다섯 리뷰 점수의 가중합이다.

초기 가중치는 0.2씩이다. 마지막 weight 층을 0으로 초기화하므로 첫 backward에서는 해당 층부터 가중치 학습을 시작한다. 학습 전 실제 1+8 배치에서 차원, 패딩 불변성, 가중치 합, 세 모듈의 gradient와 ColBERT 고정 상태를 검사한다.

In [ ]:
class AttentionScorer(nn.Module):
    """각 리뷰 토큰이 해당 후보 상품의 토큰들만 보는 단방향 attention이다."""

    def __init__(self):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            EMBED_DIM, NUM_ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True
        )
        self.match = nn.Sequential(
            nn.Linear(4 * EMBED_DIM, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, 1)
        )
        self.weight = nn.Sequential(
            nn.Linear(4 * EMBED_DIM, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 1)
        )
        nn.init.zeros_(self.weight[-1].weight)
        nn.init.zeros_(self.weight[-1].bias)

    def forward(self, reviews, review_mask, docs, doc_mask):
        """[B,5,L,64]와 [B,C,M,64]에서 상품 점수 [B,C], 가중치·리뷰 점수 [B,C,5]를 만든다."""
        b, c, m, dim = docs.shape
        _, r, length, _ = reviews.shape
        raw = reviews[:, None].expand(b, c, r, length, dim)
        kv = docs.reshape(b * c, m, dim)
        # query 위치끼리 attention하지 않으므로 5L행을 한 호출로 처리해도 각 리뷰를 따로 처리한 것과 같다.
        # 같은 후보의 K/V를 공유하고, 토큰별 H를 구한 뒤 원래 리뷰 축을 즉시 되살린다.
        h, _ = self.attention(
            raw.reshape(b * c, r * length, dim),
            kv,
            kv,
            key_padding_mask=~doc_mask.reshape(b * c, m),
            need_weights=False,
        )
        h = h.reshape(b, c, r, length, dim)
        z = torch.cat((raw, h, (raw - h).abs(), raw * h), dim=-1)
        mask = review_mask[:, None, :, :]
        per_token = self.match(z).squeeze(-1)
        review_scores = per_token.masked_fill(~mask, 0).sum(-1, dtype=torch.float32)
        lengths = mask.sum(-1, keepdim=True).clamp_min(1)
        pooled = z.masked_fill(~mask.unsqueeze(-1), 0).sum(-2, dtype=torch.float32) / lengths
        logits = self.weight(pooled).squeeze(-1).float()
        alpha = torch.softmax(logits, dim=-1)
        return (alpha * review_scores).sum(-1), alpha, review_scores


def review_tensors(records, *, trim=True):
    """고정 특징을 새 텐서에 모아 GPU로 한 번 옮긴다. 학습용 autograd는 Scorer에서 시작한다."""
    masks = np.stack([r['mask'] for r in records])
    # 한 배치에서 모든 리뷰가 무효인 오른쪽 열만 생략한다. 첫 토큰의 특별 토큰 마스크는 그대로 둔다.
    end = int(np.flatnonzero(masks.any(axis=(0, 1)))[-1]) + 1 if trim else masks.shape[-1]
    qs = torch.from_numpy(np.stack([r['query'][:, :end] for r in records])).to(DEVICE)
    qm = torch.from_numpy(masks[:, :, :end]).to(DEVICE)
    return qs, qm


def candidate_scores(model, q, qm, bundle, groups, size):
    """후보 조각의 원래 순서로 출력을 이어 붙인다. 학습 시 9개 모두가 같은 CE 분모에 들어간다."""
    scores, weights, per_review = [], [], []
    for start in range(0, groups.shape[1], size):
        ids = groups[:, start : start + size]
        masks = bundle['mask'][ids]
        width = int(np.flatnonzero(masks.any(axis=(0, 1)))[-1]) + 1
        d = torch.from_numpy(bundle['emb'][ids, :width].astype(np.float32)).to(DEVICE)
        dm = torch.from_numpy(masks[:, :, :width]).to(DEVICE)
        with amp_context():
            if torch.is_grad_enabled() and size < groups.shape[1]:
                out = checkpoint(model, q, qm, d, dm, use_reentrant=False, preserve_rng_state=True)
            else:
                out = model(q, qm, d, dm)
        scores.append(out[0])
        weights.append(out[1])
        per_review.append(out[2])
    return torch.cat(scores, 1), torch.cat(weights, 1), torch.cat(per_review, 1)


def listwise(scores):
    """각 쿼리의 0번 열이 Gold인 1+8 그룹에 temperature를 적용한다. 쿼리별 loss를 반환한다."""
    labels = torch.zeros(scores.shape[0], device=scores.device, dtype=torch.long)
    return F.cross_entropy(scores.float() / RANK_TEMPERATURE, labels, reduction='none')


def gradient_norm(module):
    """gradient clipping 직전 모듈별 L2 norm을 기록한다."""
    norms = [p.grad.detach().float().norm() for p in module.parameters() if p.grad is not None]
    return torch.stack(norms).norm().item() if norms else 0.0


def frozen_versions(encoder):
    """학습에서 제외한 모든 인코더 파라미터의 버전을 추적한다. in-place 갱신은 버전을 바꾼다."""
    if encoder.training or any(p.requires_grad or p.grad is not None for p in encoder.parameters()):
        raise RuntimeError('ColBERT가 완전히 고정되지 않았습니다.')
    return tuple((n, p._version) for n, p in encoder.named_parameters())


def preflight(model, encoder, record, docs, bundle, size):
    """실제 Gold 1+8의 크기·패딩·균등 초기화·gradient를 학습 시작 전 Scorer 복사본에서 확인한다."""
    versions = frozen_versions(encoder)
    # 시작 가중치와 dropout RNG를 건드리지 않도록 별도 모델을 eval 모드에서 역전파한다.
    probe = deepcopy(model).eval()
    q, qm = review_tensors([record], trim=False)
    assert q.shape == (1, 5, 72, EMBED_DIM) and qm.dtype == torch.bool
    doc_ids = docs[None, :]
    assert doc_ids.shape == (1, 9) and bundle['mask'][doc_ids].shape == (1, 9, 112)
    scores, alpha, _ = candidate_scores(probe, q, qm, bundle, doc_ids, size)
    assert scores.shape == (1, 9) and alpha.shape == (1, 9, 5) and torch.isfinite(scores).all()
    torch.testing.assert_close(alpha, torch.full_like(alpha, 0.2))
    torch.testing.assert_close(alpha.sum(-1), torch.ones_like(scores))
    # 하나의 후보에 대해 리뷰/상품의 마지막 토큰을 mask한 뒤, 해당 입력값을 바꿔도 점수가 같아야 한다.
    idx = doc_ids[:, :1]
    d = torch.from_numpy(bundle['emb'][idx].astype(np.float32)).to(DEVICE)
    dm = torch.from_numpy(bundle['mask'][idx]).to(DEVICE)
    qm_test, dm_test = qm.clone(), dm.clone()
    qm_test[:, :, -1] = False
    dm_test[:, :, -1] = False
    with torch.no_grad():
        original = probe(q, qm_test, d, dm_test)
        changed = probe(
            q.masked_fill(~qm_test.unsqueeze(-1), 7),
            qm_test,
            d.masked_fill(~dm_test.unsqueeze(-1), 9),
            dm_test,
        )
        for a, b in zip(original, changed):
            torch.testing.assert_close(a, b)
    listwise(scores).mean().backward()
    norms = {k: gradient_norm(getattr(probe, k)) for k in ('attention', 'match', 'weight')}
    if any(not math.isfinite(v) or v <= 0 for v in norms.values()) or frozen_versions(encoder) != versions:
        raise RuntimeError(f'gradient 흐름 또는 ColBERT 고정 상태를 확인하세요. {norms}')
    return norms

## 6. 학습과 메모리

쿼리마다 9개 점수를 모두 모아 정답 열 0에 CE를 적용한다. 기본 AdamW 업데이트는 쿼리 4개다. OOM 시 같은 난수 상태와 같은 후보로 업데이트 전체를 재시도한다. 후보 수를 줄여 처리할 때 activation checkpointing을 사용한다. 배치마다 모듈별 gradient norm, 점수 평균·표준편차, 리뷰별 가중치 범위와 entropy를 기록한다.

In [ ]:
def rng_state():
    """Python·NumPy·torch와 현재 CUDA 장치의 난수 상태를 안전한 기본 자료형으로 저장한다."""
    algorithm, keys, position, has_gauss, gaussian = np.random.get_state()
    return dict(
        python=random.getstate(),
        numpy=[algorithm, keys.tolist(), position, has_gauss, gaussian],
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state(DEVICE) if DEVICE.type == 'cuda' else None,
    )


def restore_rng(state):
    """OOM 재시도와 epoch 재개 시 dropout까지 같은 난수 위치에서 이어 간다."""
    random.setstate(state['python'])
    name, keys, pos, flag, val = state['numpy']
    np.random.set_state((name, np.asarray(keys, dtype=np.uint32), pos, flag, val))
    torch.set_rng_state(state['torch'])
    if DEVICE.type == 'cuda' and state['cuda'] is not None:
        torch.cuda.set_rng_state(state['cuda'], DEVICE)


def retry_candidates(operation, size, candidates, log):
    """후보 micro-batch만 절반씩 줄인다. 동일한 후보 목록을 항상 처음부터 전부 채점한다."""
    while True:
        try:
            return operation(size['candidates'])
        except torch.cuda.OutOfMemoryError:
            if DEVICE.type != 'cuda' or size['candidates'] == 1:
                raise
        gc.collect()
        torch.cuda.empty_cache()
        size['candidates'] = max(1, min(size['candidates'], candidates) // 2)
        log_line(log, f'OOM retry | candidate micro-batch {size["candidates"]} | pool {ANN_POOL}', True)


class ScoreStats:
    """최종 상품 점수와 후보별 5개 가중치의 통계를 누적한다. 점수 표준편차는 모집단 기준이다."""

    def __init__(self):
        self.n = 0
        self.total = self.squared = self.entropy = 0.0
        self.low, self.high = np.full(5, np.inf), np.full(5, -np.inf)

    def update(self, scores, alpha):
        s = scores.detach().float().cpu().numpy().ravel().astype(np.float64)
        a = alpha.detach().float().cpu().numpy().reshape(-1, 5).astype(np.float64)
        if not np.isfinite(s).all() or not np.isfinite(a).all():
            raise FloatingPointError('Scorer 점수/가중치에 비유한 값이 있습니다.')
        self.n += len(s)
        self.total += s.sum()
        self.squared += np.dot(s, s)
        self.entropy -= (a * np.log(a.clip(1e-30))).sum()
        self.low = np.minimum(self.low, a.min(0))
        self.high = np.maximum(self.high, a.max(0))

    def merge(self, other):
        self.n += other.n
        self.total += other.total
        self.squared += other.squared
        self.entropy += other.entropy
        self.low = np.minimum(self.low, other.low)
        self.high = np.maximum(self.high, other.high)

    def metrics(self):
        mean = float(self.total / self.n)
        out = dict(
            score_mean=mean,
            score_std=math.sqrt(max(0.0, self.squared / self.n - mean**2)),
            weight_entropy=float(self.entropy / self.n),
            weight_min=float(self.low.min()),
            weight_max=float(self.high.max()),
        )
        for j, (lo, hi) in enumerate(zip(self.low, self.high), 1):
            out[f'weight_{j}_min'], out[f'weight_{j}_max'] = float(lo), float(hi)
        return out


def train_epoch(model, opt, epoch, table, bundle, size, batch_path, log):
    """실효 배치의 쿼리 수로 CE를 정규화한다. 마지막 짧은 배치와 OOM 재시도에서도 분모는 같다."""
    model.train()
    order = list(range(len(TRAIN)))
    random.Random(SEED + epoch).shuffle(order)
    effective = TRAIN_QUERY_BATCH * GRAD_ACCUM_STEPS
    total, loss_sum, counts, step_rows = ScoreStats(), 0.0, Counter(), []
    for step, start in enumerate(range(0, len(order), effective), 1):
        window = order[start : start + effective]
        groups = []
        composition = Counter(
            dict.fromkeys(
                (
                    'maxsim_hard',
                    'ann_upper_half',
                    'ann_lower_half',
                    'catalog_random',
                    'category_same',
                    'category_sibling',
                    'category_fallback_random',
                ),
                0,
            )
        )
        for idx in window:
            docs, observed = train_group(idx, epoch, table)
            groups.append(docs)
            composition.update(observed)
        groups = np.stack(groups)
        state = rng_state()

        def backward_step(candidate_batch):
            # 이전 실패의 부분 gradient를 제거하고 이 업데이트에 해당하는 쿼리 전체를 다시 계산한다.
            opt.zero_grad(set_to_none=True)
            restore_rng(state)
            subtotal, stats = 0.0, ScoreStats()
            for offset in range(0, len(window), TRAIN_QUERY_BATCH):
                block = window[offset : offset + TRAIN_QUERY_BATCH]
                q, qm = review_tensors([table[i] for i in block])
                scores, alpha, _ = candidate_scores(
                    model, q, qm, bundle, groups[offset : offset + len(block)], candidate_batch
                )
                losses = listwise(scores)
                (losses.sum() / len(window)).backward()
                subtotal += losses.detach().sum().item()
                stats.update(scores, alpha)
            return subtotal, stats

        subtotal, stats = retry_candidates(backward_step, size, 1 + NEGATIVE_COUNT, log)
        norms = {
            f'{name}_grad_norm': gradient_norm(getattr(model, name))
            for name in ('attention', 'match', 'weight')
        }
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True)
        opt.step()
        opt.zero_grad(set_to_none=True)
        total.merge(stats)
        counts.update(composition)
        loss_sum += subtotal
        record = dict(
            epoch=epoch,
            batch=step,
            queries=len(window),
            ranking_loss=subtotal / len(window),
            candidate_batch=size['candidates'],
            **stats.metrics(),
            **norms,
            **{f'negative_{k}': v for k, v in composition.items()},
        )
        step_rows.append(record)
        log_line(
            log,
            f'epoch {epoch} | batch {step}/{math.ceil(len(order) / effective)} | '
            f'rank loss {record["ranking_loss"]:.5f} | score {record["score_mean"]:.4f} ± {record["score_std"]:.4f} | '
            f'entropy {record["weight_entropy"]:.4f} | grad(attn,match,weight) '
            f'{norms["attention_grad_norm"]:.4g},{norms["match_grad_norm"]:.4g},{norms["weight_grad_norm"]:.4g}',
            step == 1 or step % 100 == 0 or start + len(window) == len(order),
        )
    batches = pd.DataFrame(step_rows)
    neg_cols = [k for k in batches if k.startswith('negative_')]
    batches[neg_cols] = batches[neg_cols].fillna(0).astype(int)
    atomic_file(batch_path, lambda p: batches.to_csv(p, index=False))
    if sum(counts.values()) != NEGATIVE_COUNT * len(order):
        raise RuntimeError('학습에서 사용한 오답 수가 맞지 않습니다.')
    log_line(log, f'epoch {epoch} | TRAIN 실제 negative 수 {dict(counts)}', True)
    return dict(
        ranking_loss=loss_sum / len(order),
        n_queries=len(order),
        **total.metrics(),
        **{k: float(batches[k].mean()) for k in norms},
        **{f'negative_{k}': v for k, v in counts.items()},
    )

## 7. VAL에서 동일한 900개 비교

각 쿼리의 후보를 끝까지 채점하고 **900개 전체에서 한 번** 정렬해 Top-100·200을 읽는다. 동점에서는 두 방법 모두 원래 ANN 순서를 유지한다. ANN 밖 정답은 전체 Recall에서 0이다. 조건부 Recall과 평균 정답 순위는 ANN에 정답이 있는 쿼리에서 계산한다. VAL loss도 이 쿼리에 한해 매 epoch 동일한 Gold 1개와 ANN 내부 오답 8개를 사용한다. `val_loss_queries`가 그 분모다.

채점 시간은 각 방법에서 상품 특징을 읽어 900개 점수를 얻는 구간이다. CUDA 연산 완료를 기다려 측정한다. MaxSim 시간은 캐시 생성 때 한 번 측정한다. 예시 쿼리에는 후보별 가중치, 리뷰 점수와 유효 길이를 함께 저장한다. 가중치는 이 모델이 해당 상품을 채점할 때 학습한 내부 값이다.

In [ ]:
@torch.inference_mode()
def validate_epoch(model, epoch, table, bundle, size, log):
    """같은 900개 전체를 각각 정렬한다. Gold가 풀 밖이면 두 순위를 비워 두고 recall 분모에 포함한다."""
    model.eval()
    stats, details, examples = ScoreStats(), [], []
    for i, row in enumerate(VAL):
        r = table[i]
        docs = r['ann']
        gold = ITEM_TO_IDX[row['item_id']]
        q, qm = review_tensors([r])
        sync_device()
        t0 = time.perf_counter()

        def score_all(batch):
            return candidate_scores(model, q, qm, bundle, docs[None, :], batch)

        scores, alpha, review_scores = retry_candidates(score_all, size, len(docs), log)
        sync_device()
        elapsed = time.perf_counter() - t0
        if scores.shape != (1, ANN_POOL):
            raise RuntimeError('900개 전체 점수를 모으기 전에 정렬할 수 없습니다.')
        stats.update(scores, alpha)
        s = scores[0].float().cpu().numpy()
        a = alpha[0].cpu().numpy()
        rs = review_scores[0].cpu().numpy()
        base_order = np.argsort(-r['maxsim'], kind='stable')
        new_order = np.argsort(-s, kind='stable')
        positions = {int(d): j for j, d in enumerate(docs)}
        g = positions.get(gold)
        present = g is not None
        base_rank = int(np.flatnonzero(base_order == g)[0]) + 1 if present else None
        new_rank = int(np.flatnonzero(new_order == g)[0]) + 1 if present else None
        vloss = None
        if present:
            val_docs, _ = sample_group(
                r, gold, random.Random(f'{SEED}|val|{row["query_id"]}'), training=False
            )
            idx = [positions[d] for d in val_docs]
            vloss = listwise(scores[:, idx]).item()
        lengths = r['mask'].sum(-1)
        details.append(
            dict(
                epoch=epoch,
                query_id=row['query_id'],
                gold_item_id=row['item_id'],
                gold_in_ann900=present,
                gold_ann_rank=g + 1 if present else None,
                gold_maxsim_rank=base_rank,
                gold_attention_rank=new_rank,
                maxsim_hit_at_100=present and base_rank <= RERANK_TOPK,
                attention_hit_at_100=present and new_rank <= RERANK_TOPK,
                maxsim_hit_at_200=present and base_rank <= 200,
                attention_hit_at_200=present and new_rank <= 200,
                maxsim_top1_item=ITEM_IDS[docs[base_order[0]]],
                attention_top1_item=ITEM_IDS[docs[new_order[0]]],
                attention_gold_score=float(s[g]) if present else None,
                val_ranking_loss=vloss,
                maxsim_scoring_seconds=float(r['maxsim_seconds']),
                attention_scoring_seconds=elapsed,
                maxsim_top100_item_ids=json.dumps(
                    [ITEM_IDS[d] for d in docs[base_order[:RERANK_TOPK]]], ensure_ascii=False
                ),
                attention_top100_item_ids=json.dumps(
                    [ITEM_IDS[d] for d in docs[new_order[:RERANK_TOPK]]], ensure_ascii=False
                ),
                **{f'review_{j}_valid_tokens': int(n) for j, n in enumerate(lengths, 1)},
            )
        )
        if i < WEIGHT_SAMPLE_QUERIES:
            for rank, pos in enumerate(new_order, 1):
                base = dict(
                    epoch=epoch,
                    query_id=row['query_id'],
                    item_id=ITEM_IDS[docs[pos]],
                    ann_rank=int(pos) + 1,
                    attention_rank=rank,
                    attention_score=float(s[pos]),
                    is_gold=bool(pos == g),
                )
                for j in range(5):
                    base[f'alpha_{j + 1}'] = float(a[pos, j])
                    base[f'review_{j + 1}_score'] = float(rs[pos, j])
                    base[f'review_{j + 1}_valid_tokens'] = int(lengths[j])
                examples.append(base)
        if i == 0 or (i + 1) % 50 == 0 or i + 1 == len(VAL):
            log_line(log, f'epoch {epoch} | VAL {i + 1}/{len(VAL)} | 후보 {len(docs)}', True)
    df = pd.DataFrame(details)
    n_hit = int(df.gold_in_ann900.sum())
    vloss = float(df.val_ranking_loss.dropna().mean()) if n_hit else None
    metrics = dict(ranking_loss=vloss, loss_queries=n_hit, n_queries=len(df), **stats.metrics())
    return df, pd.DataFrame(examples), metrics


def stage_rows(frame, epoch):
    """조건부 Recall과 평균 순위는 두 방법 모두 정답이 ANN에 있는 동일 쿼리 집합에서 계산한다."""
    inside = frame.gold_in_ann900
    result = []
    for key, label in [('maxsim', 'Existing MaxSim'), ('attention', 'Cross-Attention')]:
        hit = frame[f'{key}_hit_at_100']
        recovered = frame.attention_hit_at_100 & ~frame.maxsim_hit_at_100
        dropped = frame.maxsim_hit_at_100 & ~frame.attention_hit_at_100
        result.append(
            dict(
                model=label,
                epoch=epoch if key == 'attention' else None,
                n_queries=len(frame),
                gold_in_ann_count=int(inside.sum()),
                ann_recall_at_900=inside.mean(),
                scorer_recall_at_100=hit.mean(),
                scorer_recall_at_200=frame[f'{key}_hit_at_200'].mean(),
                conditional_recall_at_100=hit[inside].mean(),
                mean_gold_rank_in_ann=frame.loc[inside, f'gold_{key}_rank'].mean(),
                scoring_seconds_per_query=frame[f'{key}_scoring_seconds'].mean(),
                gained_at_100=int(recovered.sum()) if key == 'attention' else None,
                lost_at_100=int(dropped.sum()) if key == 'attention' else None,
            )
        )
    return pd.DataFrame(result)


def write_validation(folder, epoch, frame, weights, metrics):
    """한 epoch의 결과를 함께 확정한다. 빈 개선·악화 집계도 0행 헤더와 실제 0개 수를 남긴다."""
    stages = stage_rows(frame, epoch)
    gains = frame.attention_hit_at_100 & ~frame.maxsim_hit_at_100
    losses = frame.maxsim_hit_at_100 & ~frame.attention_hit_at_100
    changes = []
    counts = []
    for name, mask in [('gained_at_100', gains), ('lost_at_100', losses)]:
        part = frame.loc[mask, ['query_id', 'gold_item_id', 'gold_maxsim_rank', 'gold_attention_rank']].copy()
        part.insert(0, 'change', name)
        changes.append(part)
        counts.append(dict(epoch=epoch, change=name, query_count=len(part)))
    with TemporaryDirectory(dir=folder.parent) as tmp:
        stage = Path(tmp) / 'epoch'
        stage.mkdir()
        frame.to_csv(stage / 'query_diagnostics.csv', index=False)
        weights.to_csv(stage / 'review_weights.csv', index=False)
        stages.to_csv(stage / 'stages.csv', index=False)
        pd.concat(changes, ignore_index=True).to_csv(stage / 'query_changes.csv', index=False)
        pd.DataFrame(counts).to_csv(stage / 'change_counts.csv', index=False)
        write_json(stage / 'metrics.json', dict(training_id=TRAINING_ID, epoch=epoch, metrics=metrics))
        stage.rename(folder)

## 8. 체크포인트, 재개, 비교

첫 설정은 `data304_outputs/colbert_attention_reranker/`에 저장한다. 설정이 다른 실행은 같은 폴더의 `runs/<실험 ID>/`에 보관한다. 고정 특징 캐시는 공유한다. 체크포인트는 검증 전에 확정하므로 검증 도중 끊겨도 같은 epoch 모델에서 이어 간다.

| 파일 | 내용 |
|---|---|
| `checkpoints/epoch_XX.pt` | Scorer, AdamW, RNG, epoch, 설정, 고정 ColBERT 경로와 해시 |
| `training_progress.log` · `logs/batches_epoch_XX.csv` | 배치 손실, gradient, 가중치, 실제 오답 종류별 수 |
| `epoch_losses.csv` · `stage_summary.csv` | TRAIN·VAL 손실, 검색 성능, 시간, 최고 epoch |
| `query_diagnostics.csv` | 모든 완료 epoch의 쿼리별 순위·정답 점수·Top-100 |
| `validation/epoch_XX/` | 쿼리 원본, 개선·악화 ID와 개수, 예시 쿼리의 후보별 리뷰 가중치 |
| `validation/best_epoch.json` | Recall@100이 가장 높은 epoch와 측정된 MaxSim 대비 선택 결과 |

참고값은 사용자가 제공한 ANN Recall@900 **87.43%**, MaxSim Recall@100 **72.29%**다. 마지막 표에는 이번 후보에서 재측정한 값만 들어간다. Attention epoch끼리 Recall@100 동률이면 이른 epoch를 선택하고, 최고 Attention 성능이 측정된 MaxSim과 같거나 낮으면 MaxSim을 선택한다.

In [ ]:
def experiment_folder(config, training_id):
    """첫 설정은 요청한 루트에, 설정이 다른 재실험은 별도 runs 하위에 보존한다."""
    with file_lock(ROOT / '.register.lock'):
        marker = ROOT / 'experiment.json'
        first = read_json(marker)['training_id'] if marker.is_file() else training_id
        folder = ROOT if first == training_id else ROOT / 'runs' / training_id[:20]
        folder.mkdir(parents=True, exist_ok=True)
        manifest = folder / 'experiment.json'
        if manifest.is_file():
            previous = read_json(manifest)
            if previous['training_id'] != training_id or previous['config'] != config:
                raise ValueError(f'기존 실험과 설정이 다릅니다. {manifest}')
        else:
            atomic_file(manifest, lambda p: write_json(p, dict(training_id=training_id, config=config)))
        for sub in ('checkpoints', 'validation', 'logs'):
            (folder / sub).mkdir(exist_ok=True)
        return folder


def save_epoch(path, epoch, model, opt, train, size):
    """학습한 Scorer와 AdamW 상태를 저장한다. 고정 인코더는 원본 경로와 SHA-256으로 참조한다."""
    state = dict(
        epoch=epoch,
        scorer_state_dict=model.state_dict(),
        optimizer_state_dict=opt.state_dict(),
        config=CONFIG,
        source_colbert_checkpoint=str(SOURCE),
        source_colbert_sha256=SOURCE_SHA,
        training_id=TRAINING_ID,
        rng_state=rng_state(),
        train_metrics=train,
        candidate_batch=size['candidates'],
    )
    if path.exists():
        raise FileExistsError(f'이미 완료한 epoch 파일입니다. {path}')
    atomic_file(path, lambda p: torch.save(state, p))


def restore_epoch(path, epoch, model, opt, size):
    """모델·AdamW·난수와 마지막 성공한 배치 크기를 같은 실험에서만 복원한다."""
    state = load_pt(path)
    if (
        state['epoch'] != epoch
        or state['training_id'] != TRAINING_ID
        or state['config'] != CONFIG
        or state['source_colbert_sha256'] != SOURCE_SHA
        or not state['optimizer_state_dict']['state']
    ):
        raise ValueError(f'체크포인트 출처 또는 optimizer 상태가 다릅니다. {path}')
    model.load_state_dict(state['scorer_state_dict'], strict=True)
    opt.load_state_dict(state['optimizer_state_dict'])
    restore_rng(state['rng_state'])
    size['candidates'] = state['candidate_batch']
    return state['train_metrics']


def collect_reports(folder):
    """완료된 epoch 전부를 모아 R@100 기준 최고 epoch를 선택하고 기존 MaxSim 유지 여부를 기록한다."""
    summaries, losses, queries = [], [], []
    for epoch in range(1, EPOCHS + 1):
        result = folder / 'validation' / f'epoch_{epoch:02d}'
        if not result.is_dir():
            continue
        state = load_pt(folder / 'checkpoints' / f'epoch_{epoch:02d}.pt')
        val = read_json(result / 'metrics.json')
        if (
            state['training_id'] != TRAINING_ID
            or val['training_id'] != TRAINING_ID
            or val['epoch'] != epoch
            or state['epoch'] != epoch
        ):
            raise ValueError(f'검증 결과와 체크포인트가 다릅니다. {result}')
        q = pd.read_csv(result / 'query_diagnostics.csv', dtype={'query_id': str, 'gold_item_id': str})
        if q.query_id.tolist() != [str(r['query_id']) for r in VAL] or not q.epoch.eq(epoch).all():
            raise ValueError(f'VAL 순서가 달라졌습니다. {result}')
        stages = pd.read_csv(result / 'stages.csv')
        summaries.append(stages if not summaries else stages[stages.model.eq('Cross-Attention')])
        losses.append(
            dict(
                epoch=epoch,
                **{f'train_{k}': v for k, v in state['train_metrics'].items()},
                **{f'val_{k}': v for k, v in val['metrics'].items()},
            )
        )
        queries.append(q)
    summary = pd.concat(summaries, ignore_index=True)
    attention = summary[summary.model.eq('Cross-Attention')]
    best = attention.sort_values(['scorer_recall_at_100', 'epoch'], ascending=[False, True]).iloc[0]
    baseline = summary[summary.model.eq('Existing MaxSim')].iloc[0]
    wins = bool(best.scorer_recall_at_100 > baseline.scorer_recall_at_100)
    summary['best_attention_epoch'] = summary.model.eq('Cross-Attention') & summary.epoch.eq(best.epoch)
    summary['selected_method'] = summary.best_attention_epoch if wins else summary.model.eq('Existing MaxSim')
    for name, table in [
        ('stage_summary.csv', summary),
        ('epoch_losses.csv', pd.DataFrame(losses)),
        ('query_diagnostics.csv', pd.concat(queries, ignore_index=True)),
    ]:
        atomic_file(folder / name, lambda p: table.to_csv(p, index=False))
    selected = dict(
        training_id=TRAINING_ID,
        best_attention_epoch=int(best.epoch),
        best_attention_recall_at_100=float(best.scorer_recall_at_100),
        measured_maxsim_recall_at_100=float(baseline.scorer_recall_at_100),
        selected_method='Cross-Attention' if wins else 'Existing MaxSim',
        best_attention_checkpoint=str(folder / 'checkpoints' / f'epoch_{int(best.epoch):02d}.pt'),
    )
    atomic_file(folder / 'validation' / 'best_epoch.json', lambda p: write_json(p, selected))
    return summary


def run_experiment():
    """고정 특징 준비 → gradient 검사 → 새 Scorer 학습 → 동일 ANN 후보 비교를 순서대로 수행한다."""
    global CONFIG, TRAINING_ID
    encoder = load_encoder()
    (ROOT / 'logs').mkdir(exist_ok=True)
    with (ROOT / 'logs' / 'feature_preparation.log').open('a', encoding='utf-8') as startup:
        bundle, docs = open_frozen_features(encoder, startup)
    features = dict(
        version=1,
        colbert_epoch=COLBERT_EPOCH,
        weights_sha256=SOURCE_SHA,
        index=INDEX_SPEC,
        ivf_sha256=sha256(IVF_DIR / 'token_ivf.index'),
        doc_embeddings_sha256=sha256(docs / 'doc_embeddings.npy'),
        doc_masks_sha256=sha256(docs / 'doc_masks.npy'),
        tokenizer=TOKEN_SPEC,
        split_sha256=sha256(SPLIT_PATH),
        gold_sha256=sha256(GOLD_PATH),
        ann_nprobe=ANN_NPROBE,
        ann_per_token=ANN_PER_TOKEN,
        ann_pool=ANN_POOL,
        query_precision='bfloat16' if USE_BF16 else 'float32',
        torch=str(torch.__version__),
        faiss=str(faiss.__version__),
        device=torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'cpu',
    )
    CONFIG = dict(
        version=1,
        features=features,
        architecture='one_way_attention_match_sum_and_review_weight',
        dim=EMBED_DIM,
        heads=NUM_ATTENTION_HEADS,
        attention_dropout=ATTENTION_DROPOUT,
        match_hidden=128,
        weight_hidden=64,
        mlp_dropout=0.1,
        temperature=RANK_TEMPERATURE,
        seed=SEED,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        grad_clip=MAX_GRAD_NORM,
        negatives={'maxsim': 4, 'ann_strata': 2, 'random': 1, 'category': 1},
        train_query_batch=TRAIN_QUERY_BATCH,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        candidate_batch=SCORER_CANDIDATE_BATCH,
        rerank_topk=RERANK_TOPK,
        val_query_ids=[r['query_id'] for r in VAL],
        weight_sample_queries=WEIGHT_SAMPLE_QUERIES,
    )
    TRAINING_ID = signature(CONFIG)
    folder = experiment_folder(CONFIG, TRAINING_ID)
    log_path = folder / 'training_progress.log'
    print('실험 폴더', folder, '\n진행 로그', log_path, flush=True)
    with (
        file_lock(folder / '.training.lock', blocking=False),
        log_path.open('a', buffering=1, encoding='utf-8') as log,
    ):
        log_line(
            log,
            f'run {TRAINING_ID[:16]} | frozen epoch {COLBERT_EPOCH} | index {IVF_DIR} | docs {docs}',
            True,
        )
        train_cache = prepare_queries('train', TRAIN, encoder, bundle, features, log)
        val_cache = prepare_queries('val', VAL, encoder, bundle, features, log)
        # 이후 Scorer는 캐시만 읽는다. 고정 모델은 CPU로 옮겨 GPU 메모리를 확보하고 버전을 보존한다.
        encoder.cpu()
        del bundle['index'], bundle['token2doc']
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        frozen = frozen_versions(encoder)
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        model = AttentionScorer().to(DEVICE)
        opt = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False
        )
        assert {id(p) for g in opt.param_groups for p in g['params']} == {id(p) for p in model.parameters()}
        size = {'candidates': SCORER_CANDIDATE_BATCH}
        group, _ = train_group(0, 1, train_cache)
        norms = retry_candidates(
            lambda s: preflight(model, encoder, train_cache[0], group, bundle, s), size, len(group), log
        )
        log_line(log, f'preflight 통과 | 초기 리뷰 가중치 0.2 × 5 | 모듈별 gradient {norms}', True)
        completed = [
            e for e in range(1, EPOCHS + 1) if (folder / 'checkpoints' / f'epoch_{e:02d}.pt').exists()
        ]
        if completed != list(range(1, len(completed) + 1)):
            raise ValueError('중간 epoch 체크포인트가 누락됐습니다.')
        for epoch in range(1, EPOCHS + 1):
            cp = folder / 'checkpoints' / f'epoch_{epoch:02d}.pt'
            valdir = folder / 'validation' / f'epoch_{epoch:02d}'
            if cp.is_file():
                train = restore_epoch(cp, epoch, model, opt, size)
                log_line(log, f'epoch {epoch} | Scorer + AdamW + RNG 복원', True)
            else:
                train = train_epoch(
                    model,
                    opt,
                    epoch,
                    train_cache,
                    bundle,
                    size,
                    folder / 'logs' / f'batches_epoch_{epoch:02d}.csv',
                    log,
                )
                if frozen_versions(encoder) != frozen:
                    raise RuntimeError('고정 ColBERT 파라미터가 변경됐습니다.')
                save_epoch(cp, epoch, model, opt, train, size)
                log_line(log, f'epoch {epoch} | 체크포인트 저장 {cp}', True)
            if not valdir.is_dir():
                detail, weights, metrics = validate_epoch(model, epoch, val_cache, bundle, size, log)
                write_validation(valdir, epoch, detail, weights, metrics)
            valid = read_json(valdir / 'metrics.json')['metrics']
            report = collect_reports(folder)
            stage = report[report.model.eq('Cross-Attention') & report.epoch.eq(epoch)].iloc[0]
            log_line(
                log,
                f'epoch {epoch} | train loss {train["ranking_loss"]:.5f} | val loss {valid["ranking_loss"]} '
                f'({valid["loss_queries"]}/{len(VAL)}) | ANN R@900 {stage.ann_recall_at_900:.4f} | '
                f'Attention R@100 {stage.scorer_recall_at_100:.4f}',
                True,
            )
        return report, folder

## 9. 실행

긴 후보 준비 단계에서도 완료한 쿼리 수를 기록한다. 셀에 출력된 `training_progress.log` 경로에서 `tail -f`로 진행을 볼 수 있다.

In [ ]:
results, experiment_dir = run_experiment()
display(results[['model','epoch','ann_recall_at_900','scorer_recall_at_100','scorer_recall_at_200',
                 'best_attention_epoch','selected_method']])
print('선택 결과', read_json(experiment_dir/'validation'/'best_epoch.json'), flush=True)